# SWAN-SF Decomposition LSTM Whole-Sequence Model

In [27]:
# Configure paths, random seed, and core hyperparameters
from pathlib import Path
import numpy as np
import random
import sys, os

sys.path.append(os.path.abspath("../scripts"))

from data_utils import find_project_root, load_sequence_data
from preprocessing import compute_feature_stats, normalize_sequence_data
from metrics import compute_metrics, sweep_thresholds, format_metrics
from losses import FocalLoss
from sequence_dataloader import create_dataloaders

PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / 'data' / 'Cleaned SWANSF Dataset'
print('Project root:', PROJECT_ROOT)
print('Data dir:', DATA_DIR)

random_seed = 42
random.seed(random_seed)
np.random.seed(random_seed)

# Use the same five strongest features as in previous experiments
SELECTED_FEATURES = ('SAVNCPP', 'ABSNJZH', 'R_VALUE', 'EPSZ', 'TOTFZ')

BATCH_SIZE = 32
NUM_EPOCHS = 25
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
HIDDEN_SIZE = 64
NUM_LAYERS = 1
DROPOUT = 0.3
EARLY_STOP_PATIENCE = 5
THRESHOLD_GRID = np.linspace(0.1, 0.9, 33)
MAX_ACCEPTABLE_FPR = 0.10
# Temporal smoothing window (in timesteps) for trend component
TREND_KERNEL_SIZE = 5

Project root: /Users/chintan/Drive/Projects/solar-flare-classification
Data dir: /Users/chintan/Drive/Projects/solar-flare-classification/data/Cleaned SWANSF Dataset


In [28]:
# Load SWAN-SF sequence splits and normalize
import torch
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence

X_train_all, y_train_all, X_val_all, y_val_all, X_test_all, y_test_all = load_sequence_data(
    DATA_DIR,
    SELECTED_FEATURES,
    )

print('Train shape:', X_train_all.shape)
print('Validation shape:', X_val_all.shape)
print('Test shape:', X_test_all.shape)

feature_mean, feature_std = compute_feature_stats(X_train_all)

X_train_all = normalize_sequence_data(X_train_all, feature_mean, feature_std)
X_val_all = normalize_sequence_data(X_val_all, feature_mean, feature_std)
X_test_all = normalize_sequence_data(X_test_all, feature_mean, feature_std)

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print('Using device:', device)
print('Num selected attributes:', len(SELECTED_FEATURES))

Train shape: (77865, 60, 5)
Validation shape: (19899, 60, 5)
Test shape: (331185, 60, 5)
Using device: mps
Num selected attributes: 5


In [29]:
# Create full-sequence dataloaders
train_loader, val_loader, test_loader = create_dataloaders(
    X_train_all, y_train_all,
    X_val_all, y_val_all,
    X_test_all, y_test_all,
    batch_size=BATCH_SIZE,
    )

pos_count = float((y_train_all == 1).sum())
neg_count = float((y_train_all == 0).sum())
pos_weight_value = neg_count / pos_count if pos_count > 0 else 1.0
print(f'Train samples: {len(y_train_all)} | pos={int(pos_count)} | neg={int(neg_count)} | pos_weight={pos_weight_value:.2f}')

Train samples: 77865 | pos=37873 | neg=39992 | pos_weight=1.06


In [30]:
# Decomposition LSTM model, training loop, and evaluation helpers
class DecompositionLSTM(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, num_layers: int, dropout: float, kernel_size: int):
        super().__init__()
        # Depthwise 1D convolution for per-feature smoothing (trend component)
        padding = kernel_size // 2
        self.trend_conv = nn.Conv1d(
            in_channels=input_dim,
            out_channels=input_dim,
            kernel_size=kernel_size,
            padding=padding,
            groups=input_dim,
            bias=False,
        )
        # Initialize as simple moving average
        with torch.no_grad():
            self.trend_conv.weight.fill_(1.0 / kernel_size)
        # LSTM over concatenated trend + residual
        self.lstm = nn.LSTM(
            input_size=input_dim * 2,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x, lengths):
        # x: (batch, seq_len, input_dim)
        x_perm = x.permute(0, 2, 1)  # (batch, input_dim, seq_len)
        trend = self.trend_conv(x_perm)  # (batch, input_dim, seq_len)
        trend = trend.permute(0, 2, 1)   # (batch, seq_len, input_dim)
        # Residual captures short-term deviations from the smoothed trend.
        resid = x - trend
        # Decomposition step: concatenate trend and residual before LSTM encoding.
        decomp = torch.cat([trend, resid], dim=-1)  # (batch, seq_len, 2 * input_dim)
        packed = pack_padded_sequence(decomp, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed)
        # Use the last layer's hidden state as summary
        h_last = h_n[-1]  # (batch, hidden_dim)
        logits = self.fc(self.dropout(h_last)).squeeze(1)
        return logits


model = DecompositionLSTM(len(SELECTED_FEATURES), HIDDEN_SIZE, NUM_LAYERS, DROPOUT, TREND_KERNEL_SIZE).to(device)

#criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight_value))  # BCE baseline
criterion = FocalLoss(alpha=0.75, gamma=2.0)  # Focal Loss

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)


def train_one_epoch():
    model.train()
    total = 0.0
    for seqs, labels, lengths in train_loader:
        seqs, labels, lengths = seqs.to(device), labels.to(device), lengths.to(device)
        optimizer.zero_grad()
        loss = criterion(model(seqs, lengths), labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total += loss.item() * seqs.size(0)
    return total / len(train_loader.dataset)


def predict_scores(loader):
    model.eval()
    labels, scores = [], []
    with torch.no_grad():
        for seqs, batch_labels, lengths in loader:
            seqs, lengths = seqs.to(device), lengths.to(device)
            logits = model(seqs, lengths)
            scores.append(torch.sigmoid(logits).cpu().numpy())
            labels.append(batch_labels.numpy())
    return np.concatenate(labels), np.concatenate(scores)

In [31]:
# Fit Decomposition LSTM with validation monitoring
best_state = None
best_val = {'tss': -np.inf}
best_threshold = 0.5
epochs_without_gain = 0
for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_one_epoch()
    val_labels, val_scores = predict_scores(val_loader)
    val_best, _ = sweep_thresholds(val_labels, val_scores, max_fpr=MAX_ACCEPTABLE_FPR)
    if val_best['tss'] > best_val['tss'] + 1e-4:
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        best_val = val_best
        best_threshold = val_best['threshold']
        epochs_without_gain = 0
    else:
        epochs_without_gain += 1
    scheduler.step(val_best['tss'])
    print(f"Epoch {epoch:02d} | loss={train_loss:.4f} | val_tss={val_best['tss']:.3f} | thr={val_best['threshold']:.2f} | val_fpr={val_best['fpr']:.3f}")
    if epochs_without_gain >= EARLY_STOP_PATIENCE:
        print('Early stopping triggered.')
        break

if best_state is not None:
    model.load_state_dict(best_state)
print('Best validation metrics:', format_metrics(best_val, decimals=4))
print('Best threshold:', round(float(best_threshold), 4))

Epoch 01 | loss=0.0152 | val_tss=0.908 | thr=0.70 | val_fpr=0.033
Epoch 02 | loss=0.0136 | val_tss=0.911 | thr=0.65 | val_fpr=0.036
Epoch 03 | loss=0.0133 | val_tss=0.909 | thr=0.60 | val_fpr=0.047
Epoch 04 | loss=0.0130 | val_tss=0.919 | thr=0.60 | val_fpr=0.048
Epoch 05 | loss=0.0126 | val_tss=0.902 | thr=0.60 | val_fpr=0.048
Epoch 06 | loss=0.0123 | val_tss=0.915 | thr=0.55 | val_fpr=0.046
Epoch 07 | loss=0.0118 | val_tss=0.906 | thr=0.60 | val_fpr=0.046
Epoch 08 | loss=0.0110 | val_tss=0.907 | thr=0.60 | val_fpr=0.040
Epoch 09 | loss=0.0107 | val_tss=0.895 | thr=0.55 | val_fpr=0.051
Early stopping triggered.
Best validation metrics: {'threshold': 0.6, 'roc_auc': 0.9917, 'pr_auc': 0.9906, 'accuracy': 0.9594, 'precision': 0.952, 'recall': 0.9673, 'f1': 0.9596, 'tpr': 0.9673, 'fpr': 0.0483, 'tss': 0.919, 'hss': 0.9189, 'tn': 9516.0, 'fp': 483.0, 'fn': 324.0, 'tp': 9576.0}
Best threshold: 0.6


In [32]:
# Evaluate on validation/test splits using the best threshold and show as table
import pandas as pd

val_labels_final, val_scores_final = predict_scores(val_loader)
val_metrics = compute_metrics(val_labels_final, val_scores_final, threshold=best_threshold)

test_labels, test_scores = predict_scores(test_loader)
test_metrics = compute_metrics(test_labels, test_scores, threshold=best_threshold)
test_constrained, _ = sweep_thresholds(test_labels, test_scores, max_fpr=MAX_ACCEPTABLE_FPR)

rows = [
    ("Validation @ best threshold", val_metrics),
    ("Test @ best threshold", test_metrics),
    ("Test best sweep (FPR-constrained)", test_constrained),
]

table_rows = []
for label, metrics in rows:
    table_rows.append(
        {
            "Split / Setting": label,
            "Threshold": float(metrics["threshold"]),
            "ROC-AUC": float(metrics["roc_auc"]),
            "PR-AUC": float(metrics["pr_auc"]),
            "TPR": float(metrics["tpr"]),
            "FPR": float(metrics["fpr"]),
            "TSS": float(metrics["tss"]),
            "HSS": float(metrics["hss"]),
        }
    )

metrics_table = pd.DataFrame(table_rows).round(4)
metrics_table

,Split / Setting,Threshold,ROC-AUC,PR-AUC,TPR,FPR,TSS,HSS
0,Validation @ best threshold,0.60,0.9917,0.9906,0.9673,0.0483,0.9190,0.9189
1,Test @ best threshold,0.60,0.9642,0.4045,0.9285,0.1107,0.8178,0.2156
2,Test best sweep (FPR-constrained),0.65,0.9642,0.4045,0.8924,0.0891,0.8032,0.2490


In [33]:
# Save decomposition LSTM test probabilities for ensemble notebook
import os
os.makedirs("../outputs", exist_ok=True)

test_probs = test_scores
np.save("../outputs/decomposition_lstm_probs.npy", test_probs)
print("Saved shape:", test_probs.shape)

Saved shape: (331185,)
